# LPD vs. GMM on the two-saddle sphere

Soft clustering in tangent space and time-lagged mutual information (Appendix M, Figs. 10 and 11).

In [ ]:
import glob
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.special import digamma
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = "results"  # output folder for figures
DATA_DIR = "data"  # output folder for intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
plt.style.use("tableau-colorblind10")
plt.rcParams.update(
    {
        "figure.autolayout": True,
        "font.size": 10,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "font.family": "serif",
        "font.serif": "cmr10",
        "mathtext.fontset": "cm",
        "axes.formatter.use_mathtext": True,
    }
)

## Simulation

In [ ]:
# Divergence-free flow on the sphere from the stream function (Appendix I.4)
def velocity_on_sphere(x):
    theta = np.arccos(x[2])
    phi = np.arctan2(x[1], x[0])

    sin_t, cos_t = np.sin(theta), np.cos(theta)

    dpsi_dphi = -2 * sin_t**2 * np.sin(2 * phi)
    dpsi_dtheta = 2 * sin_t * cos_t * np.cos(2 * phi)

    v_theta = -dpsi_dphi / (sin_t + 1e-12)
    v_phi = dpsi_dtheta

    e_theta = np.array([cos_t * np.cos(phi), cos_t * np.sin(phi), -sin_t])
    e_phi = np.array([-np.sin(phi), np.cos(phi), 0.0])

    return v_theta * e_theta + v_phi * e_phi


def simulate_sphere(T=500, dt=0.01, sigma=0.05, seed=None):
    if seed is not None:
        np.random.seed(seed)

    N = int(T / dt)
    x = np.array([1.0, 0.0, 0.0])
    traj = np.zeros((N + 1, 3))
    traj[0] = x

    dW = np.sqrt(dt) * np.random.randn(N, 3)

    for k in range(N):
        v = velocity_on_sphere(x)

        P = np.eye(3) - np.outer(x, x)
        noise = sigma * P @ dW[k]

        x = x + v * dt + noise
        x = x / np.linalg.norm(x)

        traj[k + 1] = x

    return traj


def finite_difference_vectors(x):

    dx = np.diff(x, axis=0)

    X = dx[:-1]
    Y = dx[1:]

    return X.T, Y.T

In [ ]:
traj = simulate_sphere(T=500, dt=0.1, sigma=0.01)
X = traj.T

X_fd, Y_fd = finite_difference_vectors(X.T)

## LPD

In [ ]:
# Offline LPD (Algorithm 2), GPU implementation
def lpd_offline_torch(
    X_np,
    Y_np,
    k=3,
    Nz_max=100,
    a_z=0.1,
    a_W=0.2,
    a_M=0.2,
    eps=1e-2,
    Nmax=2000,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X = torch.tensor(X_np, dtype=torch.float32, device=device)
    Y = torch.tensor(Y_np, dtype=torch.float32, device=device)

    Nx, T = X.shape
    Ny = Y.shape[0]

    X_t = X.T
    Y_t = Y.T

    Cxx = (X @ X.T) / T
    Cyy = (Y @ Y.T) / T
    Cxy = (X @ Y.T) / T

    I_x = torch.eye(Nx, device=device)
    I_y = torch.eye(Ny, device=device)

    Cxx_inv = torch.linalg.inv(Cxx + eps * I_x)
    Cyy_inv = torch.linalg.inv(Cyy + eps * I_y)

    m = k

    def orthogonal_matrix(dim):
        Q, _ = torch.linalg.qr(torch.randn(dim, dim, device=device))
        return Q

    M = torch.eye(m, device=device)

    W = (
        orthogonal_matrix(m) @ torch.eye(m, Nx, device=device) @ orthogonal_matrix(Nx)
    ) * 0.1

    Z = torch.rand(T, m, device=device)
    Z = torch.relu(Z)
    Z = Z / torch.clamp(torch.norm(Z, dim=0, keepdim=True), min=1e-10)

    with torch.no_grad():
        for _ in range(Nmax):

            for _ in range(Nz_max):
                Z_new = (W @ X_t.T - M @ Z.T).T
                Z = a_z * Z_new + (1 - a_z) * Z
                Z = torch.relu(Z)

            Z = Z / torch.clamp(torch.norm(Z, dim=0, keepdim=True), min=1e-10)

            M_new = Z.T @ Z
            M = a_M * M_new + (1 - a_M) * M

            Czx = (Z.T @ X_t) / torch.sqrt(
                torch.tensor(T, device=device, dtype=torch.float32)
            )

            W_new = Czx @ Cxx_inv @ Cxy @ Cyy_inv @ Cxy.T @ Cxx_inv

            W = a_W * W_new + (1 - a_W) * W

    Z_out = Z.detach().cpu().numpy()
    return Z_out, M, W

In [ ]:
Z_out, M, W = lpd_offline_torch(
    X_fd, Y_fd, k=4, a_M=0.2, a_W=0.4, a_z=0.4, Nz_max=100, Nmax=3000, eps=1e-4
)

M = M.cpu().numpy()
W = W.cpu().numpy()

MW = np.linalg.inv(M + 1e-5 * np.eye(M.shape[0])) @ W
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

plt.figure(figsize=(6, 4))
plt.imshow(MW, aspect="auto", cmap="bwr")
plt.colorbar(label="Weight strength")
plt.title(r"$M^{-1}W$ from LPD")
plt.xlabel("Input dimension (x,y,z)")
plt.ylabel("Latent component")
plt.show()

plt.figure()
plt.plot(Z_out)

In [ ]:
def normalize_memberships(Z, eps=1e-12):
    """
    Normalize so that for each sample t:
        sum_k Z[k,t] = 1
    """

    Z = np.asarray(Z, dtype=float)

    col_sums = np.sum(Z, axis=1, keepdims=True)

    col_sums = np.maximum(col_sums, eps)

    return Z / col_sums

In [ ]:
Z_norm = normalize_memberships(Z_out)

print(np.allclose(np.sum(Z_norm, axis=1), 1.0))

## GMM

In [ ]:
# GMM soft assignments on the same finite differences
def gmm_cluster_memberships(X_fd, n_clusters=5, covariance_type="full", random_state=0):

    N, T = X_fd.shape

    X = X_fd.T

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    gmm = GaussianMixture(
        n_components=n_clusters,
        covariance_type=covariance_type,
        random_state=random_state,
    )

    gmm.fit(X_scaled)

    R = gmm.predict_proba(X_scaled)

    Z = R.T

    return Z, gmm

Z, gmm = gmm_cluster_memberships(X_fd, n_clusters=4)
print(Z.shape)

## Fig. 10: trajectories and learned representations

In [ ]:
plt.rcParams.update(
    {
        "font.size": 13,
        "axes.titlesize": 13,
        "axes.labelsize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "lines.linewidth": 1.6,
    }
)

t_min, t_max = 1200, 1400

fig, axes = plt.subplots(5, 1, figsize=(7, 9), sharex=True)

colors = ["seagreen", "firebrick", "darkorange", "royalblue"]
colors_gmm = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for i in range(min(3, traj.shape[1])):
    axes[0].plot(traj[:, i], label=rf"$x_{i+1}$")

axes[0].set_ylabel("Trajectory")
axes[0].set_xlim(t_min, t_max)

axes[0].legend(
    ncol=3,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    handlelength=1.2,
    columnspacing=1.0,
    handletextpad=0.3,
    borderaxespad=0.0,
)

for i in range(min(3, X_fd.shape[0])):
    axes[1].plot(X_fd[i, :], label=rf"$\delta x_{i+1}$")

axes[1].set_ylabel(r"$\delta x$")
axes[1].set_xlim(t_min, t_max)

axes[1].legend(
    ncol=3,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    handlelength=1.2,
    columnspacing=1.0,
    handletextpad=0.3,
    borderaxespad=0.0,
)

for i in range(min(4, Z_out.shape[1])):
    axes[2].plot(Z_out[:, i], color=colors[i])

axes[2].set_ylabel("LPD")
axes[2].set_xlim(t_min, t_max)

for i in range(min(4, Z.shape[0])):
    axes[3].plot(Z[i], color=colors_gmm[i])

axes[3].set_ylabel("GMM")
axes[3].set_xlim(t_min, t_max)

for i in range(min(4, Z_norm.shape[1])):
    axes[4].plot(Z_norm[:, i], color=colors[i])

axes[4].set_ylabel("Norm. LPD")
axes[4].set_xlim(t_min, t_max)
axes[4].set_xlabel("Time")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "LPD_vs_GMM_trajectories.png"),
    dpi=1000,
    bbox_inches="tight",
)
plt.show()

## Fig. 11: time-lagged mutual information

In [ ]:
# Kraskov k-NN mutual information estimator
def mi_knn(X, Y, k=5):
    """
    Kraskov MI estimator (continuous, multivariate)

    X: (T, d_x)
    Y: (T, d_y)
    """
    X = np.atleast_2d(X)
    Y = np.atleast_2d(Y)

    if X.shape[0] != Y.shape[0]:
        X = X.T
        Y = Y.T

    n = X.shape[0]

    XY = np.concatenate([X, Y], axis=1)

    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="chebyshev").fit(XY)
    distances, _ = nbrs.kneighbors(XY)
    eps = distances[:, k]

    nbrs_x = NearestNeighbors(metric="chebyshev").fit(X)
    nbrs_y = NearestNeighbors(metric="chebyshev").fit(Y)

    nx = np.empty(n)
    ny = np.empty(n)

    for i in range(n):

        nx[i] = (
            len(nbrs_x.radius_neighbors([X[i]], eps[i], return_distance=False)[0]) - 1
        )
        ny[i] = (
            len(nbrs_y.radius_neighbors([Y[i]], eps[i], return_distance=False)[0]) - 1
        )

    return digamma(k) + digamma(n) - np.mean(digamma(nx + 1) + digamma(ny + 1))


def cross_mi(Z, Y, taus, k=5):
    """
    Z, Y: shape (T, d)
    taus: array of lags
    """
    Z = np.atleast_2d(Z)
    Y = np.atleast_2d(Y)

    if Z.shape[0] != Y.shape[0]:
        Z = Z.T
        Y = Y.T

    mi_vals = []

    for tau in taus:
        if tau > 0:
            Z_tau = Z[:-tau]
            Y_tau = Y[tau:]
        elif tau < 0:
            Z_tau = Z[-tau:]
            Y_tau = Y[:tau]
        else:
            Z_tau = Z
            Y_tau = Y

        mi_vals.append(mi_knn(Z_tau, Y_tau, k=k))

    return np.array(mi_vals)

In [ ]:
taus = np.arange(1, 102, 5)

T = 2000
dt = 0.1
N = 3

sigma_list = [0.01]
N_realizations = 10

save_dir = os.path.join(DATA_DIR, "mi_results_sphere")
os.makedirs(save_dir, exist_ok=True)

results = {}

for sigma in sigma_list:
    print(f"\n Sigma = {sigma}")

    results[sigma] = []

    for r in range(N_realizations):
        print(f"\n   Realization {r+1}/{N_realizations}")

        print("   → simulating dynamics...")

        traj = simulate_sphere(T=T, dt=dt, sigma=sigma)
        X = traj.T

        print("   → finite differences...")
        X_fd, Y_fd = finite_difference_vectors(X.T)

        print("   → LPD...")
        Z_LPD, M, W = lpd_offline_torch(
            X_fd, Y_fd, k=4, a_M=0.2, a_W=0.4, a_z=0.4, Nz_max=100, Nmax=3000, eps=1e-5
        )

        print("   → GMM...")
        Z_gmm, gmm = gmm_cluster_memberships(X_fd, n_clusters=4)

        print("   → normalize LPD...")
        Z_norm = normalize_memberships(Z_LPD)

        Y_state = X[:, 1:-1]
        Y_fd = X_fd

        print("   → computing MI...")

        mis_LPD_tangent = cross_mi(Z_LPD, Y_fd.T, taus, k=5)
        mis_GMM_tangent = cross_mi(Z_gmm.T, Y_fd.T, taus, k=5)
        mis_norm_tangent = cross_mi(Z_norm, Y_fd.T, taus, k=5)

        mis_LPD_state = cross_mi(Z_LPD, Y_state.T, taus, k=5)
        mis_GMM_state = cross_mi(Z_gmm.T, Y_state.T, taus, k=5)
        mis_norm_state = cross_mi(Z_norm, Y_state.T, taus, k=5)

        result = {
            "sigma": sigma,
            "realization": r,
            "taus": taus,
            "LPD_tangent": mis_LPD_tangent,
            "GMM_tangent": mis_GMM_tangent,
            "norm_tangent": mis_norm_tangent,
            "LPD_state": mis_LPD_state,
            "GMM_state": mis_GMM_state,
            "norm_state": mis_norm_state,
        }

        results[sigma].append(result)

        file_path = os.path.join(save_dir, f"mi_sigma_{sigma}_run_{r}.npz")

        np.savez(file_path, **result)

np.savez(os.path.join(save_dir, "ALL_RESULTS.npz"), results=results)

print("\n DONE")

In [ ]:
taus = np.arange(1, 102, 5)

T = 2000
dt = 0.1

sigma_list = [0.01]
N_realizations = 4

save_dir = os.path.join(DATA_DIR, "mi_results_sphere_hard")
os.makedirs(save_dir, exist_ok=True)

results = {}


def ensure_TK(Z):
    """
    Ensure array is (T, K)
    """
    if Z.shape[0] < Z.shape[1]:
        return Z.T
    return Z


def to_one_hot_from_soft(Z):
    """
    Z: (T, K) → one-hot (T, K)
    """
    Z = ensure_TK(Z)
    labels = np.argmax(Z, axis=1)
    K = Z.shape[1]
    return np.eye(K)[labels]

for sigma in sigma_list:
    print(f"\n Sigma = {sigma}")

    results[sigma] = []

    for r in range(N_realizations):
        print(f"\n   Realization {r+1}/{N_realizations}")

        print("   → simulating dynamics...")
        traj = simulate_sphere(T=T, dt=dt, sigma=sigma)
        X = traj.T

        print("   → finite differences...")
        X_fd, Y_fd = finite_difference_vectors(X.T)

        print("   → LPD...")
        Z_LPD, M, W = lpd_offline_torch(
            X_fd, Y_fd, k=4, a_M=0.2, a_W=0.4, a_z=0.4, Nz_max=100, Nmax=3000, eps=1e-5
        )

        print("   → GMM...")
        Z_gmm, gmm = gmm_cluster_memberships(X_fd, n_clusters=4)

        Z_LPD = ensure_TK(Z_LPD)
        Z_gmm = ensure_TK(Z_gmm)

        print("   → hard clustering...")
        Z_LPD_hard = to_one_hot_from_soft(Z_LPD)
        Z_GMM_hard = to_one_hot_from_soft(Z_gmm)

        Y_state = X[:, 1:-1]
        Y_fd = X_fd

        print("   → computing MI (hard only)...")

        mis_LPD_tangent_hard = cross_mi(Z_LPD_hard, Y_fd.T, taus, k=5)
        mis_GMM_tangent_hard = cross_mi(Z_GMM_hard, Y_fd.T, taus, k=5)

        mis_LPD_state_hard = cross_mi(Z_LPD_hard, Y_state.T, taus, k=5)
        mis_GMM_state_hard = cross_mi(Z_GMM_hard, Y_state.T, taus, k=5)

        print("      shapes:", Z_LPD_hard.shape, Z_GMM_hard.shape, Y_fd.T.shape)

        result = {
            "sigma": sigma,
            "realization": r,
            "taus": taus,
            "LPD_tangent_hard": mis_LPD_tangent_hard,
            "GMM_tangent_hard": mis_GMM_tangent_hard,
            "LPD_state_hard": mis_LPD_state_hard,
            "GMM_state_hard": mis_GMM_state_hard,
        }

        results[sigma].append(result)

        file_path = os.path.join(save_dir, f"mi_hard_sigma_{sigma}_run_{r}.npz")

        np.savez(file_path, **result)

np.savez(os.path.join(save_dir, "all_results_hard.npz"), results=results)

print("\n DONE")

In [ ]:
plt.rcParams.update(
    {
        "font.size": 13,
        "axes.labelsize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
    }
)

save_dir_soft = os.path.join(DATA_DIR, "mi_results_sphere")
files_soft = glob.glob(os.path.join(save_dir_soft, "mi_sigma_*_run_*.npz"))

results = defaultdict(list)

for f in files_soft:
    data = np.load(f, allow_pickle=True)

    sigma = float(data["sigma"])
    run_id = int(data["realization"])
    taus = data["taus"]

    results[sigma].append(
        {
            "sigma": sigma,
            "realization": run_id,
            "taus": taus,
            "LPD_tangent": data["LPD_tangent"],
            "GMM_tangent": data["GMM_tangent"],
            "norm_tangent": data["norm_tangent"],
        }
    )

save_dir_hard = os.path.join(DATA_DIR, "mi_results_sphere_hard")
files_hard = glob.glob(os.path.join(save_dir_hard, "mi_hard_sigma_*_run_*.npz"))

for f in files_hard:
    data = np.load(f, allow_pickle=True)

    sigma = float(data["sigma"])
    run_id = int(data["realization"])

    results[sigma].append(
        {
            "sigma": sigma,
            "realization": run_id,
            "taus": data["taus"],
            "LPD_tangent_hard": data["LPD_tangent_hard"],
            "GMM_tangent_hard": data["GMM_tangent_hard"],
        }
    )

for sigma in results:
    results[sigma] = sorted(results[sigma], key=lambda r: r["realization"])

sigma_target = 0.01
runs = results[sigma_target]
taus = runs[0]["taus"]

nat_to_bit = 1.0 / np.log(2)

methods = [
    "LPD_tangent",
    "GMM_tangent",
    "norm_tangent",
    "LPD_tangent_hard",
    "GMM_tangent_hard",
]

method_colors = {
    "LPD_tangent": "#1f77b4",
    "GMM_tangent": "#ff7f0e",
    "norm_tangent": "#2ca02c",
    "LPD_tangent_hard": "#1f77b4",
    "GMM_tangent_hard": "#ff7f0e",
}

method_labels = {
    "LPD_tangent": "LPD (soft)",
    "GMM_tangent": "GMM (soft)",
    "norm_tangent": "Normalized (soft)",
    "LPD_tangent_hard": "LPD (hard)",
    "GMM_tangent_hard": "GMM (hard)",
}

linestyles = {
    "LPD_tangent": "-",
    "GMM_tangent": "-",
    "norm_tangent": "-",
    "LPD_tangent_hard": "--",
    "GMM_tangent_hard": "--",
}

plt.figure(figsize=(7, 4))

for method in methods:

    valid_runs = [r for r in runs if method in r]
    if len(valid_runs) == 0:
        continue

    all_runs = np.array([r[method] for r in valid_runs]) * nat_to_bit

    mean = all_runs.mean(axis=0)
    std = all_runs.std(axis=0)

    plt.errorbar(
        taus,
        mean,
        yerr=std,
        color=method_colors[method],
        linestyle=linestyles[method],
        marker="o",
        markersize=3.5,
        capsize=3,
        elinewidth=1.2,
        label=method_labels[method],
    )

plt.xlabel(r"Prediction horizon $h$")
plt.ylabel("Mutual Information (bits)")

plt.legend(frameon=False, ncol=2)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "mi_vs_tau.png"), dpi=1000, bbox_inches="tight")
plt.show()